In [1]:
# pp.ipynb - preprocess count data and save into formatted adata files.

In [2]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [3]:
seed_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/seed/rdr_starsolo/HCC3N_600spot.spotxgene.starsolo.h5ad'
bowtie_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/align/bowtie2/matrix/HCC3N_simu.spotxgene.starsolo.h5ad"
cellranger_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim-cs_screadsim/rdr_starsolo/HCC3N_simu.spotxgene.starsolo.h5ad"

out_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/rs_benchmark/rdr/analysis/cs_bowtie2/base/pp"

In [4]:
normal_cell_type = "normal"
tumor_cell_type = "tumor"

# Load Data

In [5]:
os.makedirs(out_dir, exist_ok = True)

### Seed data (STARsolo)

In [6]:
seed = ad.read_h5ad(seed_fn)
seed

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 600 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [7]:
seed.var['feature'] = seed.var['gene']
seed = seed[:, ~seed.var["feature"].duplicated(keep = False)].copy()
seed.obs.index = seed.obs["cell"]
seed.var.index = seed.var["feature"]
seed

AnnData object with n_obs × n_vars = 600 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

In [8]:
seed.X.dtype

dtype('int64')

### scReadSim-bowtie2 (STARsolo)

In [9]:
bowtie = ad.read_h5ad(bowtie_fn)
bowtie

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [10]:
bowtie.var['feature'] = bowtie.var['gene']
bowtie = bowtie[:, ~bowtie.var["feature"].duplicated(keep = False)].copy()
bowtie.obs.index = bowtie.obs["cell"]
bowtie.var.index = bowtie.var["feature"]
bowtie

AnnData object with n_obs × n_vars = 1200 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

In [11]:
bowtie.X.dtype

dtype('int64')

### Only keep chr1-22

In [12]:
target_chroms = [str(i) for i in range(1, 23)]
bowtie = bowtie[:, bowtie.var['chrom'].isin(target_chroms)].copy()
bowtie

AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

### CellRanger (STARsolo)

In [13]:
cellranger = ad.read_h5ad(cellranger_fn)
cellranger

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [14]:
cellranger.var['feature'] = cellranger.var['gene']
cellranger = cellranger[:, ~cellranger.var["feature"].duplicated(keep = False)].copy()
cellranger.obs.index = cellranger.obs["cell"]
cellranger.var.index = cellranger.var["feature"]
cellranger

AnnData object with n_obs × n_vars = 1200 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

In [15]:
cellranger.X.dtype

dtype('int64')

## Use intersect genes

In [16]:
adata_list = [seed, bowtie, cellranger]

genes = None
for i, adata in enumerate(adata_list):
    if i == 0:
        genes = adata.var["feature"].to_numpy()
    else:
        genes = np.intersect1d(genes, adata.var["feature"])
genes.shape

(32272,)

### Use the same order of genes

In [17]:
seed = seed[:, seed.var["feature"].isin(genes)].copy()
print(seed)

bowtie = bowtie[:, seed.var.index].copy()
print(bowtie)

cellranger = cellranger[:, seed.var.index].copy()
print(cellranger)

AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'


## Split by cell type

In [18]:
seed_normal = seed[seed.obs["cell_type"] == normal_cell_type, :].copy()
print(seed_normal)

bowtie_normal = bowtie[bowtie.obs["cell_type"] == normal_cell_type, :].copy()
print(bowtie_normal)

bowtie_tumor = bowtie[bowtie.obs["cell_type"] == tumor_cell_type, :].copy()
print(bowtie_tumor)

cellranger_normal = cellranger[cellranger.obs["cell_type"] == normal_cell_type, :].copy()
print(cellranger_normal)

cellranger_tumor = cellranger[cellranger.obs["cell_type"] == tumor_cell_type, :].copy()
print(cellranger_tumor)

AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'


# Save Data

### Check the order of cells and genes

In [19]:
assert np.all(bowtie_normal.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(bowtie_tumor.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(cellranger_normal.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())
assert np.all(cellranger_tumor.var["feature"].to_numpy() == seed_normal.var["feature"].to_numpy())

In [20]:
seed_normal.var.index.name = None
bowtie_normal.var.index.name = None
bowtie_tumor.var.index.name = None
cellranger_normal.var.index.name = None
cellranger_tumor.var.index.name = None

In [21]:
seed_normal.write_h5ad(os.path.join(out_dir, "seed_normal.h5ad"), compression = 'gzip')
bowtie_normal.write_h5ad(os.path.join(out_dir, "bowtie_normal.h5ad"), compression = 'gzip')
bowtie_tumor.write_h5ad(os.path.join(out_dir, "bowtie_tumor.h5ad"), compression = 'gzip')
cellranger_normal.write_h5ad(os.path.join(out_dir, "cellranger_normal.h5ad"), compression = 'gzip')
cellranger_tumor.write_h5ad(os.path.join(out_dir, "cellranger_tumor.h5ad"), compression = 'gzip')